In [ ]:
import os
from pypdf import PdfReader

In [ ]:
all_pdf_data = [] # A list to hold text from ALL files
for filename in os.listdir("data"):
    # 2. Check if the file is a PDF (ignoring if it's .pdf or .PDF)
    if filename.lower().endswith(".pdf"):
        print(f"Processing: {filename}")

        file_path = os.path.join("data", filename)

        reader = PdfReader(file_path)
        file_text = ""
        for page in reader.pages:
            file_text += page.extract_text() + " "

        all_pdf_data.append({
            "source": filename,
            "content": file_text
        })

        first_page_text = reader.pages[0].extract_text()
        print(f"--- Document: {filename} ---")
        print(f"Total Pages: {len(reader.pages)}")
        print(f"Preview: {first_page_text[:200]}...")
        print("-" * 30)

print(f"\nSuccess! Processed {len(all_pdf_data)} documents.")

Processing: elise.pdf
--- Document: elise.pdf ---
Total Pages: 12
Preview: Can you walk me through your background? 
I’m Praj. I recently graduated from UC Berkeley with a master’s degree in information 
systems. I have a background in customer-facing analytics and consultin...
------------------------------
Processing: roku.pdf
--- Document: roku.pdf ---
Total Pages: 23
Preview: Can you walk me through your background? 
I’m Praj. I recently graduated from UC Berkeley with a master’s degree in Information 
Systems. 
I interned at Roku in 2024, working closely with UX, product,...
------------------------------

Success! Processed 2 documents.


In [ ]:
#chunking function, specify chunk size and overlap
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = [] 
    #using chunk size - overlap to create steps (repeating 50 chars) 
    #so i is 0, 450, 900..
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i + chunk_size]) #sliding window like 0 to 0+500, 450 to 450+500..
    return chunks

pdf_chunks = chunk_text(text) #call chunkcing function on extracted txt
print(f"Created {len(pdf_chunks)} chunks of text.") #print no. of chunks
print(f"Sample of Chunk 1: \n{pdf_chunks[0][:200]}...") #printing partial text from first chunk

Created 5 chunks of text.
Sample of Chunk 1: 
Can you walk me through your background? 
I’m Praj. I recently graduated from UC Berkeley with a master’s degree in Information 
Systems. 
I interned at Roku in 2024, working closely with UX, product,...


In [ ]:
all_chunks = []

for doc in all_pdf_data: # Loop through the PDFs we processed earlier
    filename = doc["source"] #to be used like a key
    content = doc["content"] #to be chunked
    
    # Chunk this specific PDF's text
    chunks = chunk_text(content)
    
    for chunk in chunks:
        # Save each chunk with its source name
        all_chunks.append({
            "source": filename,
            "text": chunk
        })

print(f"Total chunks created from all PDFs: {len(all_chunks)}")
print(f"Example chunk from {all_chunks[0]['source']}:")
print(all_chunks[0]['text'][:100])

Total chunks created from all PDFs: 135
Example chunk from elise.pdf:
Can you walk me through your background? 
I’m Praj. I recently graduated from UC Berkeley with a mas


In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Initializing vector Database (creates a 'db' folder in project)
client = chromadb.PersistentClient(path="./chroma_db")

# Defining the Embedding Function (So Chroma knows how to turn text into numbers)
# using a small model from hugging face for fast embedding
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Creating a 'Collection' (like Table)
# get_or_create prevents errors if run this cell twice
collection = client.get_or_create_collection(name="pdf_knowledge_base", embedding_function=sentence_transformer_ef)

# Preparing data for the database
# genereating unique IDs for every chunk
ids = [f"id_{i}" for i in range(len(all_chunks))]
documents = [chunk["text"] for chunk in all_chunks]
metadatas = [{"source": chunk["source"]} for chunk in all_chunks]

#Adding everything to the Vector DB
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Vector Database is ready! Added {collection.count()} chunks.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyError: 'content'